# Clean and search text · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 12. Clean and search text

Standardize labels and extract useful pieces from text columns.

Before you start: You can select columns and filter with Boolean masks.

Official reference: https://pandas.pydata.org/docs/user_guide/text.html

### M12C01. Clean spaces and preserve meaningful labels

The `.str` accessor applies text operations to each entry in a Series. `strip()` removes spaces at the beginning and end. A label map can then standardize spelling while preserving the exact labels you want.

**Read:** What does `.str.strip()` remove?

Answer: Leading and trailing whitespace from each text value.

**Predict:** What will S001 Name become?

Answer: Amira, without the surrounding spaces.

**Explain:** Will strip remove the space inside a two-word name?

Answer: No. Internal spaces are preserved.



In [ ]:
# Predict before running: What will S001 Name become?
df = raw_students()
df["Name"] = df["Name"].astype("string").str.strip()
result = df.loc[df["Student_ID"] == "S001", ["Student_ID", "Name"]]
print(result)

**Your turn:** Standardize all Campus labels as exactly RAK or Dubai. Handle case and surrounding spaces before mapping. Store Student_ID and the cleaned Campus for every source row in result.

For the check, keep these result names: `result`.

**Hint:** Use `.str.strip().str.lower()` followed by `.map({"rak": "RAK", "dubai": "Dubai"})`.

**Watch for this:** An unrecognized value passed through a dictionary map becomes missing; inspect unmatched labels.

In [ ]:
# Change this code to complete the task.
# Question: Standardize all Campus labels as exactly RAK or Dubai. Handle case and surrounding spaces before mapping. Store Student_ID and the cleaned Campus for every source row in result.
df = raw_students()
campus_text = df["Campus"].astype("string").str.strip().str.lower()
df["Campus"] = campus_text.map({"rak": "RAK", "dubai": "Dubai"})
result = df[["Student_ID", "Campus"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert set(result["Campus"]) == {"RAK", "Dubai"}
assert result.loc[1, "Campus"] == "Dubai"
assert result["Campus"].notna().all()
print("Code checks passed. Review the written explanation as well.")

S002 changes from dubai to Dubai. The abbreviation RAK remains uppercase.

**Quick check:** Would .str.title() alone preserve RAK exactly?

1. Yes
2. No, it would produce Rak
3. It would remove the whole value

Answer: 2. A known-label map is useful when abbreviations require special capitalization.

**Explain what you learned:** Standardize categories without destroying their meaning.

### M12C02. Literal text and regular expressions

`str.contains()` checks whether text contains a pattern. Use `regex=False` when you want the characters literally. With a regular expression, characters such as a dot can have special meanings.

**Read:** Why does the code set regex=False?

Answer: The dot should be an actual dot, not a pattern matching any character.

**Predict:** Which two filenames are selected?

Answer: report.csv and notes.csv.

**Explain:** What does na=False do?

Answer: It treats a missing text entry as not matching the search.



In [ ]:
# Predict before running: Which two filenames are selected?
files = pd.Series(["report.csv", "reportXcsv", "notes.csv", None, "csv.backup"], dtype="string")
mask = files.str.contains(".csv", regex=False, na=False)
result = files[mask]
print(result)

**Your turn:** Select all nonmissing filenames that begin with report. Use a regular expression and store the result.

For the check, keep these result names: `result`.

**Hint:** The ^ symbol anchors a regular expression to the start of the text.

**Watch for this:** Always be clear whether a pattern is literal text or a regular expression.

In [ ]:
# Change this code to complete the task.
# Question: Select all nonmissing filenames that begin with report. Use a regular expression and store the result.
files = pd.Series(["report.csv", "reportXcsv", "notes.csv", None, "csv.backup"], dtype="string")
mask = files.str.contains(r"^report", regex=True, na=False)
result = files[mask]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == ["report.csv", "reportXcsv"]
print("Code checks passed. Review the written explanation as well.")

Both report.csv and reportXcsv begin with report. Their extensions do not matter for this question.

**Quick check:** Which argument makes a search literal?

1. regex=False
2. na=False
3. ascending=False

Answer: 1. regex=False turns off regular-expression interpretation.

**Explain what you learned:** Choose text-search semantics deliberately.

### M12C03. Split a value into separate columns

`str.split()` separates text at a delimiter. `expand=True` creates separate columns rather than a list in each cell. Limiting splits can protect the rest of a string when a delimiter appears repeatedly.

**Read:** What character divides the email into two parts?

Answer: The @ character.

**Predict:** What will the first Username be?

Answer: amira.

**Explain:** Why use expand=True?

Answer: We want two table columns, so each piece can be selected and analyzed separately.



In [ ]:
# Predict before running: What will the first Username be?
emails = pd.Series(["amira@example.edu", "bilal@example.edu", None], dtype="string")
result = emails.str.split("@", n=1, expand=True, regex=False)
result.columns = ["Username", "Domain"]
print(result)

**Your turn:** Use the same emails. Extract Domain as a Series, then store the frequency of each known domain in result as a value_counts() Series.

For the check, keep these result names: `result`.

**Hint:** Select the second split piece with `.str[1]`, then call `.value_counts()`.

**Watch for this:** This is a string-processing exercise, not full validation of every legal email address.

In [ ]:
# Change this code to complete the task.
# Question: Use the same emails. Extract Domain as a Series, then store the frequency of each known domain in result as a value_counts() Series.
emails = pd.Series(["amira@example.edu", "bilal@example.edu", None], dtype="string")
domains = emails.str.split("@", n=1, regex=False).str[1]
result = domains.value_counts()
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.loc["example.edu"] == 2
assert len(result) == 1
print("Code checks passed. Review the written explanation as well.")

example.edu has two known occurrences. Missing domains are excluded by value_counts() by default.

**Quick check:** What object does expand=True produce here?

1. A two-column DataFrame
2. A single Python string
3. A number

Answer: 1. The split pieces become columns in a DataFrame.

**Explain what you learned:** Split text into fields before summarizing its parts.

### M12C04. Extract only values that match a structure

`str.extract()` uses capture groups to pull parts out of a matching pattern. Named groups become useful column names. A nonmatching string produces missing fields, which you should inspect.

**Read:** What do Program and Number name in the pattern?

Answer: The two captured fields, which become output columns.

**Predict:** What happens to the value bad?

Answer: Both extracted fields are missing because the whole pattern does not match.

**Explain:** Why are ^ and $ useful here?

Answer: They require the full string to match the structure, rather than accepting a matching piece inside other text.



In [ ]:
# Predict before running: What happens to the value bad?
codes = pd.Series(["AI-101", "IT-205", None, "bad"], dtype="string")
result = codes.str.extract(r"^(?P<Program>[A-Z]+)-(?P<Number>\d+)$")
print(result)

**Your turn:** Use the same codes. Extract the fields and convert Number to nullable Int64.

For the check, keep these result names: `result`.

**Hint:** Pass the extracted Number column through `pd.to_numeric(..., errors="coerce")`.

**Watch for this:** A successful pattern match checks structure, not whether the code exists in an official registry.

In [ ]:
# Change this code to complete the task.
# Question: Use the same codes. Extract the fields and convert Number to nullable Int64.
codes = pd.Series(["AI-101", "IT-205", None, "bad"], dtype="string")
result = codes.str.extract(r"^(?P<Program>[A-Z]+)-(?P<Number>\d+)$")
result["Number"] = pd.to_numeric(result["Number"], errors="coerce").astype("Int64")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert str(result["Number"].dtype) == "Int64"
assert result.loc[0, "Number"] == 101
assert result["Number"].isna().sum() == 2
print("Code checks passed. Review the written explanation as well.")

The valid numeric codes are 101 and 205. The missing and nonmatching source entries remain missing.

**Quick check:** Does extract automatically turn captured digits into integers?

1. Yes
2. No; convert the extracted text explicitly
3. Only when the group has a name

Answer: 2. Matching digits is different from choosing a numeric data type.

**Explain what you learned:** Extract structured pieces and then assign appropriate types.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates